# Masked Model — Consolidated Notebook

Multi-modal VAE experiments: model training, metrics comparison, latent analysis, masking/imputation, and doublet analysis.

## Cell 0: Configuration

In [ ]:
ADATA_PATH = '/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/adata_annotated.h5ad'
HOTSPOT_PKL = '/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/HOTSPOT/hotspot_final.pkl'
ANNOTATIONS_CSV = '/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/annotations.csv'
ABUNDANCE_LAYER = 'arcsinh'
SPATIAL_LAYER = 'HOTSPOT_top500_var'
BATCH_KEY = 'run'
MASK_FRAC = 0.1
MAX_EPOCHS = 10000
SAVE_DIR = '/home/projects/nyosef/zvise/PixelGen/PixelGen/models/'

# Set True to retrain all models even if cached versions exist
FORCE_TRAIN = False

# Doublet analysis paths (Section 6)
DATA_DIR = '/home/projects/nyosef/zvise/PxlgnProject/Data'
DOUBLETS_SEP_CACHE = '/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/doublet_separation_run'
SEPARATION_PKL_PATH = '/home/projects/nyosef/zvise/PixelGen/PixelGen/cache/results.pkl'
T_CELLS_ADATA_PATH = '/home/projects/nyosef/zvise/PixelGen/PixelGen/Data/adatas/final_adatas/t_cell_adata.h5ad'

## Cell 1: Imports

In [ ]:
import sys, os, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
from sklearn.decomposition import PCA

sys.path.insert(0, '..')
# Cached models were saved with 'PixelGen.*' module paths in their pickle;
# adding the grandparent lets torch.load resolve them.
_grandparent = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _grandparent not in sys.path:
    sys.path.insert(0, _grandparent)

from multimodalvi import MultiModalSCVI
from enums import D, AggMethod
from pxl_utils import train_model, get_model_latents
from utils import mask_adata, plot_composite_ppc, plot_model_latents, calculate_metrics
from metrics import MultiModalVIMetrics
from scvi_utils import pca_neighbors_umap, plot_losses, calc_PCA

from masked_model_utils import (
    get_default_model_kwargs,
    get_default_train_kwargs,
    load_or_train,
    run_abundance_model,
    run_spatial_model,
    run_joint_model,
    evaluate_imputation,
)

sc.settings.set_figure_params(dpi=100, frameon=False)

def get_dense(x):
    return x.toarray() if hasattr(x, 'toarray') else np.asarray(x)

---
## Section 1: Model Training
Load real data, prepare spatial features, train 4 model variants (loads from cache if available).

In [ ]:
# Load data and merge hotspot results
adata = sc.read_h5ad(ADATA_PATH)
results = pd.read_pickle(HOTSPOT_PKL)
common = adata.obs_names.intersection(results.index)
adata = adata[common].copy()
adata.obsm["HOTSPOT"] = results.loc[adata.obs_names]

# Add cell type annotations if available
if os.path.exists(ANNOTATIONS_CSV):
    annotations_df = pd.read_csv(ANNOTATIONS_CSV).set_index('component', drop=True)
    adata.obs = adata.obs.merge(annotations_df['cell_type'], how='left', left_index=True, right_index=True)

print(adata)

In [ ]:
# Feature selection: top 500 by variance * (1 - sparsity)
for layer_name in ['HOTSPOT', 'spatial_asinh']:
    if layer_name not in adata.obsm:
        continue
    df = pd.DataFrame(adata.obsm[layer_name])
    var = df.var()
    zeros = (df == 0).mean()
    score = var * (1 - zeros)
    top500 = score.sort_values(ascending=False).head(500).index
    adata.obsm[f"{layer_name}_top500_var"] = df[top500]
    print(f"{layer_name}_top500_var: {adata.obsm[f'{layer_name}_top500_var'].shape}")

# PCA on concatenated abundance + spatial
spatial_layer = SPATIAL_LAYER
sp = adata.obsm[spatial_layer]
ab = adata.layers[ABUNDANCE_LAYER]
X_concat = np.concatenate([get_dense(ab), get_dense(sp)], axis=1)
adata.obsm["pca"] = PCA(n_components=50).fit_transform(X_concat)

In [ ]:
# Train abundance-only model (cached: models/old/abundance_only)
model_ab = run_abundance_model(
    adata, ABUNDANCE_LAYER, 'abundance_only',
    batch_key=BATCH_KEY, max_epochs=MAX_EPOCHS,
    save_path=f'{SAVE_DIR}old/abundance_only',
    force_train=FORCE_TRAIN,
)

In [ ]:
# Train spatial-only model (cached: models/HOTSPOT_top500_spatial_only)
model_sp = run_spatial_model(
    adata, spatial_layer, 'spatial_only',
    batch_key=BATCH_KEY, max_epochs=MAX_EPOCHS,
    save_path=f'{SAVE_DIR}HOTSPOT_top500_spatial_only',
    force_train=FORCE_TRAIN,
)

In [ ]:
# Train joint model — shared encoder (cached: models/old/single_encoders_HOTSPOT)
model_shared = run_joint_model(
    adata, ABUNDANCE_LAYER, spatial_layer, 'joint_shared_encoder',
    method='shared_encoder', batch_key=BATCH_KEY, max_epochs=MAX_EPOCHS,
    save_path=f'{SAVE_DIR}old/single_encoders_HOTSPOT',
    force_train=FORCE_TRAIN,
)

In [ ]:
# Train joint model — weighted PoE (cached: models/old/joint_encoders_POE_HOTSPOT)
model_weighted = run_joint_model(
    adata, ABUNDANCE_LAYER, spatial_layer, 'joint_weighted_POE',
    method='weighted', batch_key=BATCH_KEY, max_epochs=MAX_EPOCHS,
    save_path=f'{SAVE_DIR}old/joint_encoders_POE_HOTSPOT',
    force_train=FORCE_TRAIN,
)

---
## Section 2: Metrics & Comparison
MultiModalVIMetrics evaluation, posterior predictive checks, scib-metrics.

In [ ]:
# Collect all trained/loaded models for evaluation
models_dict = {
    'abundance_only': model_ab,
    'spatial_only': model_sp,
    'joint_shared_encoder': model_shared,
    'joint_weighted_POE': model_weighted,
}
print(f"Models loaded: {list(models_dict.keys())}")

In [ ]:
# scib-metrics — condition
metrics_condition = MultiModalVIMetrics(
    adata, models_dict,
    pca_key='pca', batch_key=BATCH_KEY, biological_key='condition',
)
metrics_condition.run()
_ = metrics_condition.mean_modality_errors_barplot(reconstruction_mean=True)
metrics_condition.plot_scib_metrics()

In [ ]:
# scib-metrics — cell_type
if 'cell_type' in adata.obs.columns:
    metrics_celltype = MultiModalVIMetrics(
        adata, models_dict,
        pca_key='pca', batch_key=BATCH_KEY, biological_key='cell_type',
    )
    metrics_celltype.run()
    metrics_celltype.plot_scib_metrics()

In [ ]:
# Posterior predictive checks
all_metrics = []
model_names = list(models_dict.keys())
spatial_layers_candidates = ['HOTSPOT_top500_var', 'spatial_asinh_top500_var']
ab_raw = get_dense(adata.layers[ABUNDANCE_LAYER])

for model_name, model in models_dict.items():
    imputed = model.get_normalized_expression(
        adata=adata, return_mean_expression=True,
        return_l2_error=True, return_px_distrs=False, return_numpy=True,
    )

    # Abundance metrics
    if ABUNDANCE_LAYER in imputed['exprs']:
        ab_gen = get_dense(imputed['exprs'][ABUNDANCE_LAYER])
        all_metrics.extend(calculate_metrics(ab_raw, ab_gen, model_name, 'Abundance'))

    # Spatial metrics — find whichever layer the model was trained on
    for candidate in spatial_layers_candidates:
        if candidate in imputed['exprs']:
            sp_gen = get_dense(imputed['exprs'][candidate])
            if candidate in adata.obsm:
                sp_raw = get_dense(adata.obsm[candidate])
            elif candidate in adata.layers:
                sp_raw = get_dense(adata.layers[candidate])
            else:
                continue
            all_metrics.extend(calculate_metrics(sp_raw, sp_gen, model_name, 'Spatial'))
            break

metrics_df = pd.DataFrame(all_metrics)
plot_composite_ppc('Abundance', metrics_df, scatter_color='cornflowerblue', model_names=model_names)
plot_composite_ppc('Spatial', metrics_df, scatter_color='lightgreen', model_names=model_names)

---
## Section 3: Latent Analysis
UMAP visualization, Leiden clustering, differential expression, marker overlays.

In [ ]:
# Latent comparison grid — all models × (joint, abundance, spatial)
plot_model_latents(models_dict, adata, color='condition')

In [ ]:
# Color by cell_type if available
if 'cell_type' in adata.obs.columns:
    plot_model_latents(models_dict, adata, color='cell_type')

In [ ]:
# Leiden clustering on best joint latent (weighted POE)
best_latent = 'z_joint_weighted_POE_joint'
if best_latent in adata.obsm:
    sc.pp.neighbors(adata, n_neighbors=15, use_rep=best_latent)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=0.8)
    sc.pl.umap(adata, color=['leiden', 'condition'], legend_loc='on data', size=50)

In [ ]:
# Differential expression between Leiden clusters
if 'log1p' in adata.layers:
    adata_tmp = ad.AnnData(X=adata.layers['log1p'], obs=adata.obs.copy(), var=adata.var.copy())
    sc.tl.rank_genes_groups(adata_tmp, 'leiden', method="wilcoxon")
    diff_exp_df = sc.get.rank_genes_groups_df(adata_tmp, group=None)
    diff_exp_df["-log10(adjusted p-value)"] = -np.log10(diff_exp_df["pvals_adj"])
    diff_exp_df["Significant"] = diff_exp_df["pvals_adj"] < 0.01

    # Heatmap of top markers
    markers = set(
        diff_exp_df[
            (np.abs(diff_exp_df["logfoldchanges"]) > 3) & diff_exp_df["Significant"]
        ]["names"]
    )
    if markers:
        df_pivot = diff_exp_df.pivot(index="names", columns="group", values="logfoldchanges")
        df_pivot = df_pivot[df_pivot.index.isin(markers)]
        fig = sns.clustermap(df_pivot, yticklabels=True, linewidths=0.1, cmap="vlag", vmin=-5, vmax=5)
        fig.fig.set_size_inches(10, 15)

In [ ]:
# Marker gene overlays on UMAP
rel_genes = ['CD4', 'CD8', 'FMC63', 'CD44', 'CD21', 'CD137', 'CD25']
available = [g for g in rel_genes if g in adata.var_names]
if available and 'log1p' in adata.layers:
    sc.pl.umap(adata, color=available, frameon=False, size=50, layer='log1p', ncols=4)

---
## Section 4: Masking & Imputation
Mask spatial data, train masked model, evaluate imputation quality.

In [ ]:
# Create masked copy — uses mask_adata() from utils.py
spatial_key_for_masking = 'spatial_asinh' if 'spatial_asinh' in adata.obsm else SPATIAL_LAYER
adata_masked = adata.copy()
mask_adata(adata_masked, spatial_key=spatial_key_for_masking, mask_frac=MASK_FRAC)
masked_spatial_layer = f'{spatial_key_for_masking}_masked'
print(f"Masked layer: {masked_spatial_layer}")
print(f"Masked cells: {adata_masked.obs['spatial_masked'].sum()}/{adata_masked.n_obs}")

In [ ]:
# Train masked joint model (cached: models/old/masked_joint_encoders_POE)
model_masked = run_joint_model(
    adata_masked,
    ab_layer=ABUNDANCE_LAYER,
    sp_layer=masked_spatial_layer,
    model_name='masked_joint_POE',
    method='weighted',
    batch_key=BATCH_KEY,
    max_epochs=MAX_EPOCHS,
    spatial_mask_key='spatial_masked',
    save_path=f'{SAVE_DIR}old/masked_joint_encoders_POE',
    force_train=FORCE_TRAIN,
)

In [ ]:
# Evaluate imputation quality — L2 error on masked vs unmasked cells
imputation_results = evaluate_imputation(
    adata_original=adata,
    adata_masked=adata_masked,
    model=model_masked,
    spatial_key=masked_spatial_layer,
)

In [ ]:
# UMAP of imputed spatial modality
imputed = model_masked.get_normalized_expression(
    adata=adata_masked, return_mean_expression=True,
    return_l2_error=True, return_px_distrs=False, return_numpy=True,
)
imputed_spatial = imputed["exprs"][masked_spatial_layer]

adata_imputed = sc.AnnData(X=imputed_spatial, obs=adata_masked.obs.copy())
adata_imputed.obs["spatial_masked"] = (
    adata_imputed.obs["spatial_masked"].astype(bool).astype(str).astype("category")
)
sc.pp.neighbors(adata_imputed, n_neighbors=15, use_rep="X")
sc.tl.umap(adata_imputed)

color_cols = ["spatial_masked", "condition"]
if "cell_type" in adata_imputed.obs.columns:
    color_cols.append("cell_type")
sc.pl.umap(adata_imputed, color=color_cols)

In [ ]:
# Latent visualization of masked model
plot_model_latents({'masked_model': model_masked}, adata_masked, color='spatial_masked')

---
## Section 5: Doublet Analysis (Optional)
T-cell subset annotation, cell-wise colocalization via `doublet_separation.py`, combined adata with separated doublets.

> **Note:** Requires pixelator PNA data files in `DATA_DIR`. Skip if those files are not available.

In [ ]:
# T-cell subset identification and doublet annotation
t_cells = adata[adata.obs['cell_type'].isin(['CD8', 'CD4'])].copy()
t_cells.obsm['arcsinh'] = t_cells.layers['arcsinh']

sc.pp.neighbors(t_cells, n_neighbors=15, use_rep="arcsinh")
sc.tl.leiden(t_cells, resolution=0.7)
sc.tl.umap(t_cells)

rel_genes = ['CD4', 'CD8', 'FMC63', 'CD44', 'CD21', 'CD137', 'CD25']
available = [g for g in rel_genes if g in t_cells.var_names]
sc.pl.umap(t_cells, color=['leiden', 'condition'] + available, ncols=4, size=50, layer='log1p')

In [ ]:
# Annotate doublet clusters — adjust mapping based on visual inspection
cell_annotations = {
    "0": "CD8_doublets",
    "1": "Non_doublets",
    "2": "Non_doublets",
    "3": "CD8_doublets",
    "4": "CD4_doublets",
    "5": "Non_doublets",
    "6": "Non_doublets",
    "7": "Non_doublets",
}
t_cells.obs["doublets_status"] = t_cells.obs["leiden"].map(cell_annotations)
sc.pl.umap(t_cells, color=['doublets_status', 'condition'], size=50)

# Optionally save annotated T-cells
# t_cells.write_h5ad(T_CELLS_ADATA_PATH)

In [ ]:
# Doublet separation — requires PNA data files
# Uncomment and run if pixelator PNA files are available in DATA_DIR

# from pathlib import Path
# from pixelator import read
# from doublet_separation.doublet_separation import (
#     run_cellwise_coloc_analysis_to_disk, concat_abundance_to_adata,
#     add_doublets_metadata, B_CD8_logfc_dict, B_CD4_logfc_dict,
# )
#
# files = [f for f in Path(DATA_DIR).rglob('*.pxl') if f.is_file()]
# pg_data = read(files)
#
# doublets = t_cells[t_cells.obs['doublets_status'].str.startswith('CD')].copy()
# abundance_all, abundance_scaled_all, zscores_all, coloc_mean_all = (
#     run_cellwise_coloc_analysis_to_disk(
#         doublets=doublets, pg_data=pg_data, adata=doublets,
#         logfc_cd8=B_CD8_logfc_dict, logfc_cd4=B_CD4_logfc_dict,
#         out_dir=DOUBLETS_SEP_CACHE, k=2, n_sims=100, seed=123,
#         overwrite=False, checkpoint_every=50,
#     )
# )
#
# with open(SEPARATION_PKL_PATH, "wb") as f:
#     pickle.dump(dict(
#         abundance_all=abundance_all, abundance_scaled_all=abundance_scaled_all,
#         zscores_all=zscores_all, mean_scores=coloc_mean_all,
#     ), f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Load pre-computed doublet separation results and visualize
# Uncomment if results.pkl exists

# with open(SEPARATION_PKL_PATH, "rb") as f:
#     sep_results = pickle.load(f)
#
# abundance_scaled_all = sep_results["abundance_scaled_all"]
# abundance_scaled_all.fillna(0, inplace=True)
#
# # Combine separated doublets with singlets
# doublets_cells = t_cells.obs[t_cells.obs.doublets_status.str.startswith('CD')].index
# singlets = adata[~adata.obs.index.isin(doublets_cells)].copy()
# singlets.X = singlets.layers['counts']
#
# adata_combined = concat_abundance_to_adata(singlets, abundance_scaled_all)
# adata_combined = add_doublets_metadata(t_cells, adata_combined)
# adata_combined.obsm['combined_arcsinh'] = np.arcsinh(adata_combined.X / 5.0).astype(np.float32)
# adata_combined.obsm['combined_log1p'] = np.log1p(adata_combined.X).astype(np.float32)
#
# sc.pp.neighbors(adata_combined, n_neighbors=15, use_rep="combined_log1p")
# sc.tl.leiden(adata_combined, resolution=0.7)
# sc.tl.umap(adata_combined)
# sc.pl.umap(adata_combined, color=["condition", 'cell_group', 'dataset'], ncols=3)